In [ ]:
#!pip install scikit-network
#!pip install markov-clustering
#!pip install python-louvain

In [ ]:
from sklearn.cluster import SpectralClustering
import markov_clustering as mc
import community as community_louvain
from sknetwork.clustering import Louvain, get_modularity

In [ ]:
import pandas as pd
data=pd.read_csv("data2-PA.csv",header=None,names=['src','dst','dist'])
src=list(data['src'])
dst=list(data['dst'])
dis=list(data['dist'])
import networkx as nx
import random
G = nx.Graph()
for i in range(0,len(src)):
  G.add_edge(src[i], dst[i], dis=dis[i])

In [ ]:
print(len(G.nodes()))
print(len(G.edges()))

1088092
1541898


In [ ]:
A = nx.to_numpy_array(G)

In [ ]:
def generateGraph(nbNode,proba):
    G = nx.gnp_random_graph(nbNode, proba)
    for e in G.edges():
        G[e[0]][e[1]]['dist']=random.randint(1, 20)

    nx.write_graphml(G, "graph-"+str(nbNode)+".graphml")
def intersection(lst1, lst2):
    lst3 = [value for value in lst1 if value in lst2]
    return lst3
def getdistance(garph,path):
    dist=0
    for i in range(0,len(path)-1):
        dist=dist+HG[path[i]][path[i+1]]['dist']
    return dist
def graphPartitioning(G,nbPart):
    A = nx.to_numpy_array(G)
    sc = SpectralClustering(nbPart, affinity='precomputed', n_init=100)
    sc.fit(A)
    clusters=[]
    for c in range(0,nbPart):
        clusters.append([])
    allnodes=[]
    allnodes=G.nodes()
    labels=sc.labels_
    for  i,node in enumerate(G.nodes()) :
          clusters[labels[i]].append(node)
    dic_clusters={}
    for i in range(0,len(clusters)):
          dic_clusters[i+1]={'nodes':clusters[i],'frontierNodes':set(),'externalNodes':set()}
    for i in range(0,len(clusters)):
      for n in dic_clusters[i+1]['nodes']:
        edges=G.edges([n])
        for e in edges:
          a,b=e
          if a not in dic_clusters[i+1]['nodes'] and b in dic_clusters[i+1]['nodes'] :
            dic_clusters[i+1]['frontierNodes'].add(b)
            dic_clusters[i+1]['externalNodes'].add(a)
          if b not in dic_clusters[i+1]['nodes'] and a in dic_clusters[i+1]['nodes']:
            dic_clusters[i+1]['frontierNodes'].add(a)
            dic_clusters[i+1]['externalNodes'].add(b)
    for c in dic_clusters.keys():
      dic_clusters[c]['connectedClusters']=set()
    #annotate clusters to frontier cluster or bridge cluster
    for c in dic_clusters.keys():
      connected_cluster=set()
      for cc in dic_clusters.keys():
        if cc!=c:
          inter=intersection(dic_clusters[c]['nodes'],list(dic_clusters[cc]['externalNodes']))
          if len(inter)>0:
            dic_clusters[c]['connectedClusters'].add(cc)
    return dic_clusters
def buildHyperGraph(G,dic_clusters):
    #build an hypergraph
    HG = nx.Graph()
    for c in dic_clusters.keys():
      front=dic_clusters[c]['frontierNodes']
      ext=dic_clusters[c]['externalNodes']
      inter=dic_clusters[c]['nodes']
      for n in front:
        for m in ext:
          if G.has_edge(n,m) :
            dist=G[n][m]['dist']
            HG.add_edge(n,m,dist=dist,label='direct')

      for n in inter :
        for m in inter:
          if G.has_edge(n,m) and n in front and m in front:
            dist=G[n][m]['dist']
            HG.add_edge(n,m,dist=dist,label='direct')
    #get heyper link on bridge clusters
    for c in dic_clusters.keys():
      if len(dic_clusters[c]['connectedClusters'])>1:
        internalnodes=list(dic_clusters[c]['frontierNodes'])
        sn1=G.subgraph(internalnodes)
        for i in range(0,len(internalnodes)):
          for j in range(0,len(internalnodes)):
            if i> j:
              shortest_path1 = nx.shortest_path(G, source=internalnodes[i], target=internalnodes[j], weight='dist')
              dist=nx.shortest_path_length(G, source=internalnodes[i], target=internalnodes[j], weight='dist')
              HG.add_edge(internalnodes[i],internalnodes[j],dist=dist, label=shortest_path1)
    return HG
def getPathUsingHG(src,dst,HG):
    start_time = time.time()
    srcIDCluster=getclusterID(srcID,dic_clusters)
    dstIDCluster=getclusterID(dstID,dic_clusters)
    if srcIDCluster == dstIDCluster:
      cluster=G.subgraph(dic_clusters[dstIDCluster]["nodes"])
      shortest_path1 = nx.shortest_path(cluster, source=srcID, target=dstID, weight='dist')
      print("Le chemin loacl ",shortest_path1)
      end_time = time.time()
      rt1 = end_time - start_time
      print("running time using partition of the graph: ", rt1)
    else:
      #get frontier nodes of clusters (src,dst)
      F_src=dic_clusters[srcIDCluster]["frontierNodes"]
      F_dst=dic_clusters[dstIDCluster]["frontierNodes"]
      cluster1=G.subgraph(dic_clusters[srcIDCluster]["nodes"])
      cluster2=G.subgraph(dic_clusters[dstIDCluster]["nodes"])
      #get new src and dst from frontier clusters, and get the shotestpath from src and frontier nodes of the src cluster and shotestpath dst and frontier nodes of the dst cluster
      newsrc=-1
      min_dis1=sys.maxsize
      minpath1=[]
      for n in F_src:
        shortest_path=nx.shortest_path(cluster1, source=srcID, target=n, weight='dist')
        distance = nx.shortest_path_length(cluster1, source=srcID, target=n, weight='dist')
        if distance < min_dis1:
          min_dis1=distance
          newsrc=shortest_path[-1]
          minpath1=shortest_path
      newdst=-1
      min_dis2=sys.maxsize
      minpath2=[]
      for n in F_dst:
        shortest_path=nx.shortest_path(cluster2, source=n, target=dstID, weight='dist')
        distance = nx.shortest_path_length(cluster2, source=n, target=dstID, weight='dist')
        if distance < min_dis2:
          min_dis2=distance
          newdst=shortest_path[0]
          minpath2=shortest_path

      print("The new src and dst are :",newsrc, newdst)
      shortest_path=nx.shortest_path(HG, source=newsrc, target=newdst, weight='dist')
      distance = nx.shortest_path_length(HG, source=newsrc, target=newdst, weight='dist')
      end_time = time.time()
      rt1 = end_time - start_time
      print("running time using the global graph: ", rt1)

In [ ]:
generateGraph(50,0.3)

In [ ]:
G2 = nx.read_graphml("graph-50.graphml")

In [ ]:
nbPart=5
G=G2

In [ ]:
dictG=graphPartitioning(G,3)
hg=buildHyperGraph(G,dic_clusters)

NodeView(('33', '24', '28', '5', '37', '16', '44', '11', '43', '14', '19', '49', '41', '10', '0', '15', '1', '18', '4', '13', '48', '3', '25', '31', '20', '21', '2', '8', '34', '38', '7', '39', '29', '6', '30', '9', '22', '40', '32', '46', '27', '26', '47', '36', '12', '35', '42', '23', '17', '45'))

In [ ]:
A = nx.to_numpy_array(G)
sc = SpectralClustering(nbPart, affinity='precomputed', n_init=100)
sc.fit(A)

SpectralClustering(affinity='precomputed', n_clusters=5, n_init=100)

In [ ]:
A

array([[0., 0., 0., ..., 1., 0., 0.],
       [0., 0., 1., ..., 0., 1., 0.],
       [0., 1., 0., ..., 0., 0., 0.],
       ...,
       [1., 0., 0., ..., 0., 1., 1.],
       [0., 1., 0., ..., 1., 0., 0.],
       [0., 0., 0., ..., 1., 0., 0.]])

In [ ]:
clusters=[]
for c in range(0,nbPart):
    clusters.append([])
allnodes=[]
allnodes=G.nodes()
labels=sc.labels_
for  i,node in enumerate(G.nodes()) :
      clusters[labels[i]].append(node)

In [ ]:
dic_clusters={}
for i in range(0,len(clusters)):
  dic_clusters[i+1]={'nodes':clusters[i],'frontierNodes':set(),'externalNodes':set()}

In [ ]:
for i in range(0,len(clusters)):
  for n in dic_clusters[i+1]['nodes']:
    edges=G.edges([n])
    for e in edges:
      a,b=e
      if a not in dic_clusters[i+1]['nodes'] and b in dic_clusters[i+1]['nodes'] :
        dic_clusters[i+1]['frontierNodes'].add(b)
        dic_clusters[i+1]['externalNodes'].add(a)
      if b not in dic_clusters[i+1]['nodes'] and a in dic_clusters[i+1]['nodes']:
        dic_clusters[i+1]['frontierNodes'].add(a)
        dic_clusters[i+1]['externalNodes'].add(b)

In [ ]:
for c in dic_clusters.keys():
  dic_clusters[c]['connectedClusters']=set()

In [ ]:
#annotate clusters to frontier cluster or bridge cluster
for c in dic_clusters.keys():
  connected_cluster=set()
  for cc in dic_clusters.keys():
    if cc!=c:
      inter=intersection(dic_clusters[c]['nodes'],list(dic_clusters[cc]['externalNodes']))
      if len(inter)>0:
        dic_clusters[c]['connectedClusters'].add(cc)

In [ ]:
print(dic_clusters)

{1: {'nodes': ['4', '11', '12', '26', '30', '31', '33', '34', '40', '42', '47'], 'frontierNodes': {'33', '11', '4', '31', '34', '30', '40', '26', '47', '12', '42'}, 'externalNodes': {'43', '14', '24', '32', '46', '19', '20', '28', '49', '13', '48', '21', '27', '5', '37', '9', '16', '44', '38', '41', '36', '10', '7', '0', '15', '22', '39', '29', '3', '1', '18', '35', '2', '25', '8', '6'}, 'connectedClusters': {2, 3, 4, 5}}, 2: {'nodes': ['0', '7', '14', '23', '28', '32', '36', '37', '38', '41', '44'], 'frontierNodes': {'36', '23', '7', '41', '0', '28', '14', '37', '44', '32', '38'}, 'externalNodes': {'43', '31', '30', '26', '47', '46', '42', '20', '33', '19', '49', '48', '13', '17', '21', '27', '9', '5', '16', '11', '10', '4', '15', '22', '39', '40', '29', '3', '1', '18', '35', '45', '2', '34', '25', '8', '12', '6'}, 'connectedClusters': {1, 3, 4, 5}}, 3: {'nodes': ['1', '3', '6', '10', '16', '18', '19', '20', '21', '25', '35', '43', '45', '46', '48'], 'frontierNodes': {'19', '16', '10'

In [ ]:
#build an hypergraph
HG = nx.Graph()
for c in dic_clusters.keys():
  front=dic_clusters[c]['frontierNodes']
  ext=dic_clusters[c]['externalNodes']
  inter=dic_clusters[c]['nodes']
  for n in front:
    for m in ext:
      if G.has_edge(n,m) :
        dist=G[n][m]['dist']
        HG.add_edge(n,m,dist=dist,label='direct')

  for n in inter :
    for m in inter:
      if G.has_edge(n,m) and n in front and m in front:
        dist=G[n][m]['dist']
        HG.add_edge(n,m,dist=dist,label='direct')
#get heyper link on bridge clusters
for c in dic_clusters.keys():
  if len(dic_clusters[c]['connectedClusters'])>1:
    internalnodes=list(dic_clusters[c]['frontierNodes'])
    sn1=G.subgraph(internalnodes)
    for i in range(0,len(internalnodes)):
      for j in range(0,len(internalnodes)):
        if i> j:
          shortest_path1 = nx.shortest_path(G, source=internalnodes[i], target=internalnodes[j], weight='dist')
          dist=nx.shortest_path_length(G, source=internalnodes[i], target=internalnodes[j], weight='dist')
          HG.add_edge(internalnodes[i],internalnodes[j],dist=dist, label=shortest_path1)

In [ ]:
HG.nodes()

NodeView(('33', '24', '28', '5', '37', '16', '44', '11', '43', '14', '19', '49', '41', '10', '0', '15', '1', '18', '4', '13', '48', '3', '25', '31', '20', '21', '2', '8', '34', '38', '7', '39', '29', '6', '30', '9', '22', '40', '32', '46', '27', '26', '47', '36', '12', '35', '42', '23', '17', '45'))

In [ ]:
shortest_path1 = nx.shortest_path(HG, source='24', target='45', weight='dist')
print(shortest_path1)

['24', '45']


In [ ]:
srcID='1'
dstID='37'

In [ ]:
def getclusterID(nodeID,dicClusters):
  for c in dicClusters.keys():
    if nodeID in dicClusters[c]['nodes']:
      print()
      return c
  return -1
print(getclusterID(srcID,dic_clusters))
print(getclusterID(dstID,dic_clusters))
print(dic_clusters)


3

2
{1: {'nodes': ['4', '11', '12', '26', '30', '31', '33', '34', '40', '42', '47'], 'frontierNodes': {'33', '11', '4', '31', '34', '30', '40', '26', '47', '12', '42'}, 'externalNodes': {'43', '14', '24', '32', '46', '19', '20', '28', '49', '13', '48', '21', '27', '5', '37', '9', '16', '44', '38', '41', '36', '10', '7', '0', '15', '22', '39', '29', '3', '1', '18', '35', '2', '25', '8', '6'}, 'connectedClusters': {2, 3, 4, 5}}, 2: {'nodes': ['0', '7', '14', '23', '28', '32', '36', '37', '38', '41', '44'], 'frontierNodes': {'36', '23', '7', '41', '0', '28', '14', '37', '44', '32', '38'}, 'externalNodes': {'43', '31', '30', '26', '47', '46', '42', '20', '33', '19', '49', '48', '13', '17', '21', '27', '9', '5', '16', '11', '10', '4', '15', '22', '39', '40', '29', '3', '1', '18', '35', '45', '2', '34', '25', '8', '12', '6'}, 'connectedClusters': {1, 3, 4, 5}}, 3: {'nodes': ['1', '3', '6', '10', '16', '18', '19', '20', '21', '25', '35', '43', '45', '46', '48'], 'frontierNodes': {'19', '16'

In [ ]:
import sys
import time
# usage of the global algo
#test if src and dst are in the same sub graph, then we pply shotestpath on cluster
# else
#
start_time = time.time()
srcIDCluster=getclusterID(srcID,dic_clusters)
dstIDCluster=getclusterID(dstID,dic_clusters)
if srcIDCluster == dstIDCluster:
  cluster=G.subgraph(dic_clusters[dstIDCluster]["nodes"])
  shortest_path1 = nx.shortest_path(cluster, source=srcID, target=dstID, weight='dist')
  print("Le chemin loacl ",shortest_path1)
  end_time = time.time()
  rt1 = end_time - start_time
  print("running time using partition of the graph: ", rt1)
else:
  #get frontier nodes of clusters (src,dst)
  F_src=dic_clusters[srcIDCluster]["frontierNodes"]
  F_dst=dic_clusters[dstIDCluster]["frontierNodes"]
  cluster1=G.subgraph(dic_clusters[srcIDCluster]["nodes"])
  cluster2=G.subgraph(dic_clusters[dstIDCluster]["nodes"])
  #get new src and dst from frontier clusters, and get the shotestpath from src and frontier nodes of the src cluster and shotestpath dst and frontier nodes of the dst cluster
  newsrc=-1
  min_dis1=sys.maxsize
  minpath1=[]
  for n in F_src:
    shortest_path=nx.shortest_path(cluster1, source=srcID, target=n, weight='dist')
    distance = nx.shortest_path_length(cluster1, source=srcID, target=n, weight='dist')
    if distance < min_dis1:
      min_dis1=distance
      newsrc=shortest_path[-1]
      minpath1=shortest_path
  newdst=-1
  min_dis2=sys.maxsize
  minpath2=[]
  for n in F_dst:
    shortest_path=nx.shortest_path(cluster2, source=n, target=dstID, weight='dist')
    distance = nx.shortest_path_length(cluster2, source=n, target=dstID, weight='dist')
    if distance < min_dis2:
      min_dis2=distance
      newdst=shortest_path[0]
      minpath2=shortest_path

  print("The new src and dst are :",newsrc, newdst)
  shortest_path=nx.shortest_path(HG, source=newsrc, target=newdst, weight='dist')
  distance = nx.shortest_path_length(HG, source=newsrc, target=newdst, weight='dist')
  end_time = time.time()
  rt1 = end_time - start_time
  print("running time using the global graph: ", rt1)
  print(minpath1,min_dis1)
  print(minpath2,min_dis2)
  print(shortest_path,distance)




The new src and dst are : 1 37
running time using the global graph:  0.01120615005493164
['1'] 0
['37'] 0
['1', '11', '15', '37'] 6


In [ ]:
def getPathUsingHG(src,dst,HG):
    start_time = time.time()
    srcIDCluster=getclusterID(srcID,dic_clusters)
    dstIDCluster=getclusterID(dstID,dic_clusters)
    if srcIDCluster == dstIDCluster:
      cluster=G.subgraph(dic_clusters[dstIDCluster]["nodes"])
      shortest_path1 = nx.shortest_path(cluster, source=srcID, target=dstID, weight='dist')
      print("Le chemin loacl ",shortest_path1)
      end_time = time.time()
      rt1 = end_time - start_time
      print("running time using partition of the graph: ", rt1)
    else:
      #get frontier nodes of clusters (src,dst)
      F_src=dic_clusters[srcIDCluster]["frontierNodes"]
      F_dst=dic_clusters[dstIDCluster]["frontierNodes"]
      cluster1=G.subgraph(dic_clusters[srcIDCluster]["nodes"])
      cluster2=G.subgraph(dic_clusters[dstIDCluster]["nodes"])
      #get new src and dst from frontier clusters, and get the shotestpath from src and frontier nodes of the src cluster and shotestpath dst and frontier nodes of the dst cluster
      newsrc=-1
      min_dis1=sys.maxsize
      minpath1=[]
      for n in F_src:
        shortest_path=nx.shortest_path(cluster1, source=srcID, target=n, weight='dist')
        distance = nx.shortest_path_length(cluster1, source=srcID, target=n, weight='dist')
        if distance < min_dis1:
          min_dis1=distance
          newsrc=shortest_path[-1]
          minpath1=shortest_path
      newdst=-1
      min_dis2=sys.maxsize
      minpath2=[]
      for n in F_dst:
        shortest_path=nx.shortest_path(cluster2, source=n, target=dstID, weight='dist')
        distance = nx.shortest_path_length(cluster2, source=n, target=dstID, weight='dist')
        if distance < min_dis2:
          min_dis2=distance
          newdst=shortest_path[0]
          minpath2=shortest_path

      print("The new src and dst are :",newsrc, newdst)
      shortest_path=nx.shortest_path(HG, source=newsrc, target=newdst, weight='dist')
      distance = nx.shortest_path_length(HG, source=newsrc, target=newdst, weight='dist')
      end_time = time.time()
      rt1 = end_time - start_time
      print("running time using the global graph: ", rt1)

In [ ]:
getPathUsingHG('1','37',HG)

In [ ]:
HG['11']['37']

{'dist': 15, 'label': 'direct'}

In [ ]:
getdistance(HG,minpath1)